# 18.9 標準答案和教師偏好怎麼一起學？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**alpha=0應回到哪種訓練？

混合標準答案CE和教師分佈KL，既保留標註真值，也接受額外候選訊息。alpha與T²convention必須記錄，不能只寫「蒸餾loss」。

**把直覺寫成數學：**L=(1−alpha)CE+alpha·T²·KL，0≤alpha≤1；本helper內部已乘T²，外層別再乘。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import distillation_loss
from tiny_perceptron.model import masked_loss

s = torch.randn(1, 2, 4, requires_grad=True)
t = torch.randn(1, 2, 4)
y = torch.tensor([[1, 2]])
for alpha in (0.0, 0.5, 1.0):
    print(alpha, distillation_loss(s, t, y, alpha=alpha).item())
assert torch.allclose(distillation_loss(s, t, y, alpha=0.0), masked_loss(s, y))

**如何讀結果：**temperature改變梯度也改變候選分佈；調參用validation，test保留。

**只改一件事：**只移除T²縮放，記錄你改了哪種convention。
